# 05 — BM25+ Degradation Analysis

Diagnoses **why BM25+ retrieval performance changes across popularity deciles**.

Each section tests a specific hypothesis using the enriched BM25+ results DataFrame
and the `bm25_analysis` module.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path().resolve().parent.parent))
from notebooks.retrieval_eval.shared_setup import *
from src.metrics.bm25_analysis import *
from src.metrics.decile_utils import boundaries_for

BM25_KEY = 'retrieved_docs_bm25_plus'
df = results_by_strategy[BM25_KEY].copy()
boundaries = boundaries_for(DECILE_MODE, boundaries_uw, boundaries_cw)
print(f'BM25+ rows: {len(df):,}')
print(f'Decile col: {decile_col}')
print(f'Score examples: {df.iloc[0]["topk_scores"][:3]}')
print(f'Has doc_length: {"doc_length" in df.columns}')

## 1. Hit Rate & Score Gap by Decile

**Question:** Are popular documents systematically harder for BM25+ to find?

Measures hit_rate@10, and for found documents, the score gap between the
gold doc and the top-1 retrieved doc. A **large gap** means BM25+ assigns
much higher scores to competing documents.

In [ ]:
gap_df = compute_score_gap_by_decile(df, decile_col, top_k=TOP_K)
print(gap_df.to_string(index=False, float_format='%.4f'))

In [ ]:
d_idx = gap_df['decile'] + 1
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Panel A — Hit rate
ax = axes[0]
ax.bar(d_idx, gap_df['hit_rate'], color='#3498db', alpha=0.85, edgecolor='white')
ax.set_title('A. Hit Rate @10\n(gold in top-10)', fontweight='bold', fontsize=12)
ax.set_xlabel('Popularity Decile (1=Rare → 10=Famous)', fontweight='bold')
ax.set_ylabel('Fraction Found', fontweight='bold')
ax.set_xticks(d_idx)
ax.grid(axis='y', alpha=0.3)

# Panel B — Score gap (top1 - gold)
ax = axes[1]
ax.bar(d_idx, gap_df['mean_score_gap'], color='#e74c3c', alpha=0.85, edgecolor='white')
ax.set_title('B. Score Gap\n(top1_score - gold_score, found only)', fontweight='bold', fontsize=12)
ax.set_xlabel('Popularity Decile (1=Rare → 10=Famous)', fontweight='bold')
ax.set_ylabel('Mean Score Gap', fontweight='bold')
ax.set_xticks(d_idx)
ax.grid(axis='y', alpha=0.3)

# Panel C — Score ratio (gold / top1)
ax = axes[2]
ax.bar(d_idx, gap_df['mean_score_ratio'], color='#27ae60', alpha=0.85, edgecolor='white')
ax.axhline(1.0, color='grey', linestyle='--', linewidth=1)
ax.set_title('C. Score Ratio\n(gold_score / top1_score)', fontweight='bold', fontsize=12)
ax.set_xlabel('Popularity Decile (1=Rare → 10=Famous)', fontweight='bold')
ax.set_ylabel('Mean Score Ratio', fontweight='bold')
ax.set_xticks(d_idx)
ax.grid(axis='y', alpha=0.3)

fig.suptitle('BM25+ Score Gap Analysis by Popularity Decile', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
out = IMAGES_DIR / 'bm25_score_gap_by_decile.png'
fig.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'✓ Saved → {out.name}')

## 2. Popularity Displacement Analysis

**Question:** What types of documents displace the gold doc when it's not found?

Compares the gold doc's popularity to the mean popularity of retrieved docs.
If BM25+ systematically retrieves **more popular** docs, those are the
competitors pushing the gold doc out of the top-k.

In [ ]:
disp_df = compute_popularity_displacement(df, decile_col, top_k=TOP_K)
print(disp_df.to_string(index=False, float_format='%.4f'))

In [ ]:
d_idx = disp_df['decile'] + 1
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Panel A — Popularity ratio
ax = axes[0]
ax.bar(d_idx, disp_df['mean_pop_ratio'], color='#8e44ad', alpha=0.85, edgecolor='white')
ax.axhline(1.0, color='grey', linestyle='--', linewidth=1)
ax.set_title('A. Popularity Ratio\n(mean_retrieved / gold_pop)', fontweight='bold', fontsize=12)
ax.set_xlabel('Popularity Decile', fontweight='bold')
ax.set_ylabel('Ratio', fontweight='bold')
ax.set_xticks(d_idx)
ax.grid(axis='y', alpha=0.3)

# Panel B — # more popular retrieved
ax = axes[1]
ax.bar(d_idx, disp_df['mean_n_more_popular'], color='#c0392b', alpha=0.85, edgecolor='white')
ax.set_title('B. # Retrieved Docs MORE Popular\n(than gold, avg per question)', fontweight='bold', fontsize=12)
ax.set_xlabel('Popularity Decile', fontweight='bold')
ax.set_ylabel('Mean Count', fontweight='bold')
ax.set_xticks(d_idx)
ax.grid(axis='y', alpha=0.3)

# Panel C — # less popular retrieved
ax = axes[2]
ax.bar(d_idx, disp_df['mean_n_less_popular'], color='#2980b9', alpha=0.85, edgecolor='white')
ax.set_title('C. # Retrieved Docs LESS Popular\n(than gold, avg per question)', fontweight='bold', fontsize=12)
ax.set_xlabel('Popularity Decile', fontweight='bold')
ax.set_ylabel('Mean Count', fontweight='bold')
ax.set_xticks(d_idx)
ax.grid(axis='y', alpha=0.3)

fig.suptitle('BM25+ Popularity Displacement by Decile', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
out = IMAGES_DIR / 'bm25_popularity_displacement.png'
fig.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'✓ Saved → {out.name}')

## 3. Gold Document Rank Distribution

**Question:** When the gold doc IS found, where does it typically rank?

Shows the fraction of questions where gold appears at rank 1, 2, ..., 10
vs not found at all.  If popular docs are systematically pushed to lower
ranks, that explains the recall degradation.

In [ ]:
rank_df = compute_rank_distribution_by_decile(df, decile_col, top_k=TOP_K)
print(rank_df[['decile', 'count', 'frac_not_found', 'frac_rank_1', 'frac_rank_2', 'frac_rank_3',
               'mean_rank_found', 'median_rank_found']].to_string(index=False, float_format='%.4f'))

In [ ]:
d_idx = rank_df['decile'] + 1
rank_cols = [f'frac_rank_{k}' for k in range(1, TOP_K + 1)]
colors = plt.cm.RdYlGn_r(np.linspace(0.1, 0.9, len(rank_cols)))

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Panel A — Stacked bar of rank fractions
ax = axes[0]
bottom = np.zeros(len(rank_df))
for i, col in enumerate(rank_cols):
    vals = rank_df[col].values
    ax.bar(d_idx, vals, bottom=bottom, color=colors[i], edgecolor='white',
           linewidth=0.5, label=f'Rank {i+1}')
    bottom += vals
ax.bar(d_idx, rank_df['frac_not_found'], bottom=bottom, color='#333333',
       alpha=0.7, edgecolor='white', linewidth=0.5, label='Not found')
ax.set_title('A. Gold Rank Distribution\n(fraction at each rank)', fontweight='bold', fontsize=12)
ax.set_xlabel('Popularity Decile (1=Rare → 10=Famous)', fontweight='bold')
ax.set_ylabel('Fraction', fontweight='bold')
ax.set_xticks(d_idx)
ax.legend(fontsize=7, ncol=3, loc='upper right')
ax.grid(axis='y', alpha=0.3)

# Panel B — Mean rank when found
ax = axes[1]
valid = rank_df['mean_rank_found'].notna()
ax.bar(d_idx[valid.values], rank_df.loc[valid, 'mean_rank_found'],
       color='#e67e22', alpha=0.85, edgecolor='white')
ax.set_title('B. Mean Rank When Found\n(lower = better)', fontweight='bold', fontsize=12)
ax.set_xlabel('Popularity Decile (1=Rare → 10=Famous)', fontweight='bold')
ax.set_ylabel('Mean Rank (1-10)', fontweight='bold')
ax.set_xticks(d_idx)
ax.grid(axis='y', alpha=0.3)

fig.suptitle('BM25+ Gold Document Rank Distribution by Decile', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
out = IMAGES_DIR / 'bm25_rank_distribution.png'
fig.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'✓ Saved → {out.name}')

## 4. Competition Analysis

**Question:** How much competition does each decile face in the index?

Popular articles have far more chunks in the index.  This means:
- More competing chunks from the same popularity stratum
- More noise per query (irrelevant chunks that partially match)

The **competition index** = `n_chunks / n_questions` — higher values mean
each question faces more competing chunks.

In [ ]:
questions_per_decile = np.array(
    [int((df[decile_col] == d).sum()) for d in range(10)], dtype=float
)
comp_df = compute_competition_stats(corpus_docs, corpus_chunks, questions_per_decile)
print(comp_df.to_string(index=False, float_format='%.2f'))

In [ ]:
d_idx = comp_df['decile'] + 1
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Panel A — Chunks per doc
ax = axes[0]
ax.bar(d_idx, comp_df['chunks_per_doc'], color='#2ecc71', alpha=0.85, edgecolor='white')
ax.set_title('A. Chunks per Document\n(longer articles → more chunks)', fontweight='bold', fontsize=12)
ax.set_xlabel('Popularity Decile', fontweight='bold')
ax.set_ylabel('Chunks / Doc', fontweight='bold')
ax.set_xticks(d_idx)
ax.grid(axis='y', alpha=0.3)

# Panel B — Competition index (chunks/question)
ax = axes[1]
vals = comp_df['competition_index'].replace([np.inf], np.nan)
ax.bar(d_idx, vals, color='#e74c3c', alpha=0.85, edgecolor='white')
ax.set_title('B. Competition Index\n(n_chunks / n_questions — log scale)', fontweight='bold', fontsize=12)
ax.set_xlabel('Popularity Decile', fontweight='bold')
ax.set_ylabel('Chunks per Question', fontweight='bold')
ax.set_xticks(d_idx)
ax.set_yscale('log')
ax.grid(axis='y', alpha=0.3)

# Panel C — Random chunk hit rate
ax = axes[2]
ax.bar(d_idx, comp_df['random_chunk_hit_rate'] * 100, color='#9b59b6', alpha=0.85, edgecolor='white')
ax.set_title('C. Random Chunk Hit Rate\n(1/n_chunks %)', fontweight='bold', fontsize=12)
ax.set_xlabel('Popularity Decile', fontweight='bold')
ax.set_ylabel('Probability (%)', fontweight='bold')
ax.set_xticks(d_idx)
ax.grid(axis='y', alpha=0.3)

fig.suptitle('BM25+ Competition Analysis by Decile', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
out = IMAGES_DIR / 'bm25_competition_analysis.png'
fig.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'✓ Saved → {out.name}')

## 5. BM25 Score vs Document Length

**Question:** Does BM25+'s length normalization systematically penalise
popular (longer) documents?

BM25 uses a length-normalization parameter `b` (default 0.75) that
penalizes documents longer than the corpus average.  Since popular
articles are much longer, their per-term scores may be diluted.

Note: BM25+ indexes **chunks** (1000 chars each), so individual chunks
are similar in length.  The effect here is indirect — longer docs
produce more chunks, increasing competition.

In [ ]:
slen_df = compute_score_vs_length_by_decile(df, decile_col, top_k=TOP_K)
print(slen_df.to_string(index=False, float_format='%.2f'))

In [ ]:
d_idx = slen_df['decile'] + 1
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Panel A — Top-1 BM25 score
ax = axes[0]
ax.bar(d_idx, slen_df['mean_top1_score'], color='#3498db', alpha=0.85, edgecolor='white')
ax.set_title('A. Mean Top-1 BM25 Score\n(highest retrieved score per query)', fontweight='bold', fontsize=12)
ax.set_xlabel('Popularity Decile', fontweight='bold')
ax.set_ylabel('Mean BM25 Score', fontweight='bold')
ax.set_xticks(d_idx)
ax.grid(axis='y', alpha=0.3)

# Panel B — Gold doc length
ax = axes[1]
ax.bar(d_idx, slen_df['mean_gold_doc_length'], color='#e74c3c', alpha=0.85, edgecolor='white')
ax.set_title('B. Mean Gold Document Length\n(characters)', fontweight='bold', fontsize=12)
ax.set_xlabel('Popularity Decile', fontweight='bold')
ax.set_ylabel('Mean Length (chars)', fontweight='bold')
ax.set_xticks(d_idx)
ax.grid(axis='y', alpha=0.3)

# Panel C — Gold score when found
ax = axes[2]
valid = slen_df['mean_gold_score_if_found'].notna()
ax.bar(d_idx[valid.values], slen_df.loc[valid, 'mean_gold_score_if_found'],
       color='#27ae60', alpha=0.85, edgecolor='white')
ax.set_title('C. Mean Gold Score When Found\n(BM25 score of gold doc)', fontweight='bold', fontsize=12)
ax.set_xlabel('Popularity Decile', fontweight='bold')
ax.set_ylabel('Mean BM25 Score', fontweight='bold')
ax.set_xticks(d_idx)
ax.grid(axis='y', alpha=0.3)

fig.suptitle('BM25+ Score vs Document Length by Decile', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
out = IMAGES_DIR / 'bm25_score_vs_length.png'
fig.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'✓ Saved → {out.name}')

## 6. Retrieved Set Composition

**Question:** What popularity strata do BM25+'s retrieved documents come from?

If BM25+ retrieves documents from **the same or higher deciles** than the
gold doc, those are the competitors.  If it retrieves from **lower**
deciles, that suggests term overlap with rare-doc topics.

Key metric: `mean_retrieved_decile` — if this is above the gold decile,
BM25+ is biased toward more popular documents.

In [ ]:
comp2_df = compute_retrieved_composition_by_decile(df, decile_col, boundaries, top_k=TOP_K)
print(comp2_df.to_string(index=False, float_format='%.4f'))

In [ ]:
d_idx = comp2_df['decile'] + 1
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Panel A — Fraction same decile
ax = axes[0]
ax.bar(d_idx, comp2_df['mean_frac_same_decile'], color='#2ecc71', alpha=0.85, edgecolor='white')
ax.set_title('A. Fraction from Same Decile\n(retrieved docs matching gold decile)', fontweight='bold', fontsize=12)
ax.set_xlabel('Gold Decile', fontweight='bold')
ax.set_ylabel('Fraction', fontweight='bold')
ax.set_xticks(d_idx)
ax.grid(axis='y', alpha=0.3)

# Panel B — Fraction higher / lower
ax = axes[1]
w = 0.35
ax.bar(d_idx - w/2, comp2_df['mean_frac_higher_decile'], w, color='#e74c3c',
       alpha=0.80, edgecolor='white', label='Higher decile (more popular)')
ax.bar(d_idx + w/2, comp2_df['mean_frac_lower_decile'], w, color='#3498db',
       alpha=0.80, edgecolor='white', label='Lower decile (less popular)')
ax.set_title('B. Retrieved from Higher/Lower Deciles\n(vs gold decile)', fontweight='bold', fontsize=12)
ax.set_xlabel('Gold Decile', fontweight='bold')
ax.set_ylabel('Fraction', fontweight='bold')
ax.set_xticks(d_idx)
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.3)

# Panel C — Mean retrieved decile vs gold decile
ax = axes[2]
ax.plot([1, 10], [0, 9], 'k--', alpha=0.5, label='y = gold decile')
ax.bar(d_idx, comp2_df['mean_retrieved_decile'], color='#9b59b6', alpha=0.75,
       edgecolor='white', label='Mean retrieved decile')
ax.set_title('C. Mean Retrieved Decile\n(dashed = match with gold)', fontweight='bold', fontsize=12)
ax.set_xlabel('Gold Decile (1-based)', fontweight='bold')
ax.set_ylabel('Mean Retrieved Decile (0-based)', fontweight='bold')
ax.set_xticks(d_idx)
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.3)

fig.suptitle('BM25+ Retrieved Set Composition by Decile', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
out = IMAGES_DIR / 'bm25_retrieved_composition.png'
fig.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'✓ Saved → {out.name}')

## 7. BM25 Score vs Gold Popularity (Per-Question Scatter)

**Question:** Is there a continuous relationship between the gold doc's
popularity and the BM25+ top-1 score?

This reveals whether the degradation is gradual or threshold-based.

In [ ]:
df_valid = df.dropna(subset=['doc_length']).copy()
df_valid['log_pop'] = np.log10(df_valid['popularity_avg'].clip(lower=1e-1))

# Extract top-1 scores
def _top1(row):
    s = row.get('topk_scores')
    if s is None or len(s) == 0:
        return np.nan
    v = float(s[0])
    return v if not np.isnan(v) else np.nan

df_valid['top1_score'] = df_valid.apply(_top1, axis=1)
df_valid = df_valid.dropna(subset=['top1_score'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel A — Scatter: top1 score vs log popularity
ax = axes[0]
colors = plt.cm.viridis(df_valid[decile_col] / 9.0)
ax.scatter(df_valid['log_pop'], df_valid['top1_score'], s=3, alpha=0.25, c=colors)
ax.set_xlabel('log10(Popularity)', fontweight='bold')
ax.set_ylabel('Top-1 BM25 Score', fontweight='bold')
ax.set_title('A. Top-1 BM25 Score vs Gold Popularity', fontweight='bold', fontsize=12)
ax.grid(alpha=0.3)

# Add binned means
bins = np.linspace(df_valid['log_pop'].min(), df_valid['log_pop'].max(), 25)
df_valid['pop_bin'] = pd.cut(df_valid['log_pop'], bins=bins, labels=False)
binned = df_valid.groupby('pop_bin').agg(
    x=('log_pop', 'mean'),
    y=('top1_score', 'mean'),
    ci=('top1_score', lambda s: 1.96 * s.std() / np.sqrt(len(s)) if len(s) > 1 else 0),
).dropna()
ax.errorbar(binned['x'], binned['y'], yerr=binned['ci'], fmt='-o', color='red',
             markersize=4, linewidth=2, capsize=3, label='Binned mean ± 95% CI')
ax.legend(fontsize=9)

# Panel B — Scatter: gold doc length vs top1 score
ax = axes[1]
ax.scatter(df_valid['doc_length'], df_valid['top1_score'], s=3, alpha=0.25, c=colors)
ax.set_xlabel('Gold Document Length (chars)', fontweight='bold')
ax.set_ylabel('Top-1 BM25 Score', fontweight='bold')
ax.set_title('B. Top-1 BM25 Score vs Gold Doc Length', fontweight='bold', fontsize=12)
ax.set_xscale('log')
ax.grid(alpha=0.3)

bins_len = np.logspace(
    np.log10(df_valid['doc_length'].clip(lower=1).min()),
    np.log10(df_valid['doc_length'].max()), 25,
)
df_valid['len_bin'] = pd.cut(df_valid['doc_length'], bins=bins_len, labels=False)
binned_len = df_valid.groupby('len_bin').agg(
    x=('doc_length', 'mean'),
    y=('top1_score', 'mean'),
    ci=('top1_score', lambda s: 1.96 * s.std() / np.sqrt(len(s)) if len(s) > 1 else 0),
).dropna()
ax.errorbar(binned_len['x'], binned_len['y'], yerr=binned_len['ci'], fmt='-o', color='red',
             markersize=4, linewidth=2, capsize=3, label='Binned mean ± 95% CI')
ax.legend(fontsize=9)

fig.suptitle('BM25+ Score vs Popularity & Length (Per-Question)', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
out = IMAGES_DIR / 'bm25_score_scatter.png'
fig.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'✓ Saved → {out.name}')

## 8. Master Diagnostic Table

Combines all analyses into one summary table for quick comparison.

In [ ]:
diag = compute_bm25_diagnostic_table(
    df, decile_col, corpus_docs, corpus_chunks, boundaries, top_k=TOP_K,
)
# Select key columns for display
display_cols = [
    'decile', 'n_docs', 'n_chunks', 'chunks_per_doc',
    'hit_rate', 'mean_score_gap', 'mean_score_ratio',
    'mean_pop_ratio', 'mean_n_more_popular',
    'mean_top1_score', 'mean_gold_doc_length',
]
display_cols = [c for c in display_cols if c in diag.columns]
diag_display = diag[display_cols].copy()
diag_display['decile'] += 1
print(diag_display.to_string(index=False, float_format='%.3f'))

out = IMAGES_DIR / 'bm25_diagnostic_table.csv'
diag_display.to_csv(out, index=False, float_format='%.4f')
print(f'\n✓ Saved → {out.name}')